# Survey Demos from Paper 2

Demonstrates cross-silo vs cross-device settings, non-IID taxonomy, client dropouts, communication costs, and client drift.

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from src.utils import seed_everything, load_mnist, evaluate, count_params
from src.models import TwoNN
from src.fl import fedavg_round, client_update
from src.partition import load_partition, feature_skew_partition

seed_everything(42)
os.makedirs('results', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
(train_x, train_y), (test_x, test_y) = load_mnist('data')

Compare cross-silo (K=10, C=1.0) and cross-device (K=100, C=0.1) settings.

In [2]:
iid_part = load_partition('data/iid_k100.npz')
client_data_100 = {c: (train_x[idx], train_y[idx]) for c, idx in iid_part.items()}
silo_data = {i: (train_x[i*6000:(i+1)*6000], train_y[i*6000:(i+1)*6000]) for i in range(10)}

records_settings = []
rounds = 6

seed_everything(42)
m_silo = TwoNN().to(device)
for r in range(1, rounds + 1):
    fedavg_round(m_silo, silo_data, list(range(10)), epochs=1, batch_size=50, lr=0.05, device=device)
    loss, acc = evaluate(m_silo, test_x, test_y, device=device)
    records_settings.append({'setting': 'Cross-silo (K=10, C=1.0)', 'round': r, 'test_acc': acc, 'test_loss': loss})

seed_everything(42)
m_device = TwoNN().to(device)
for r in range(1, rounds + 1):
    perm = torch.randperm(100)[:10].tolist()
    fedavg_round(m_device, client_data_100, perm, epochs=1, batch_size=50, lr=0.05, device=device)
    loss, acc = evaluate(m_device, test_x, test_y, device=device)
    records_settings.append({'setting': 'Cross-device (K=100, C=0.1)', 'round': r, 'test_acc': acc, 'test_loss': loss})

pd.DataFrame(records_settings).to_csv('results/survey_settings.csv', index=False)
print('Settings comparison finished.')

Settings comparison finished.


Evaluate non-IID heterogeneity types: label skew, quantity skew, and feature skew.

In [3]:
records_types = []
dir_part = load_partition('data/dirichlet_a05.npz')
qty_part = load_partition('data/quantity_skew.npz')
feat_part, feat_x = feature_skew_partition(train_x, train_y, num_clients=100, seed=42)

type_splits = {
    'Label skew': {c: (train_x[idx], train_y[idx]) for c, idx in dir_part.items()},
    'Quantity skew': {c: (train_x[idx], train_y[idx]) for c, idx in qty_part.items()},
    'Feature skew': {c: (feat_x[idx], train_y[idx]) for c, idx in feat_part.items()}
}

for skew_name, c_dict in type_splits.items():
    seed_everything(42)
    m = TwoNN().to(device)
    for r in range(1, rounds + 1):
        perm = torch.randperm(100)[:10].tolist()
        fedavg_round(m, c_dict, perm, epochs=2, batch_size=20, lr=0.05, device=device)
        loss, acc = evaluate(m, test_x, test_y, device=device)
        records_types.append({'skew_type': skew_name, 'round': r, 'test_acc': acc, 'test_loss': loss})
    print(f'{skew_name} final accuracy: {acc:.4f}')

pd.DataFrame(records_types).to_csv('results/survey_noniid_types.csv', index=False)

Label skew final accuracy: 0.8942


Quantity skew final accuracy: 0.9288


Feature skew final accuracy: 0.5259


Evaluate partial participation with client dropout probabilities p in 0.0, 0.2, and 0.5.

In [4]:
records_drop = []
for p_drop in [0.0, 0.2, 0.5]:
    seed_everything(42)
    m = TwoNN().to(device)
    for r in range(1, rounds + 1):
        perm = torch.randperm(100)[:10].tolist()
        active = [c for c in perm if np.random.rand() >= p_drop]
        if not active:
            active = [perm[0]]
        fedavg_round(m, client_data_100, active, epochs=1, batch_size=20, lr=0.05, device=device)
        loss, acc = evaluate(m, test_x, test_y, device=device)
        records_drop.append({'drop_prob': f'p = {p_drop}', 'round': r, 'test_acc': acc, 'test_loss': loss})
    print(f'Drop p={p_drop} final accuracy: {acc:.4f}')

pd.DataFrame(records_drop).to_csv('results/survey_dropout.csv', index=False)

Drop p=0.0 final accuracy: 0.9019


Drop p=0.2 final accuracy: 0.8968


Drop p=0.5 final accuracy: 0.8954


Compute communication cost and track client drift via cosine similarity.

In [5]:
params = count_params(TwoNN())
bytes_per_client_round = 2 * params * 4
df_set = pd.read_csv('results/survey_settings.csv')
df_set['megabytes'] = df_set['round'] * 10 * bytes_per_client_round / (1024 * 1024)
df_set.to_csv('results/survey_communication.csv', index=False)

def measure_drift(c_dict, rounds=4):
    seed_everything(42)
    m = TwoNN().to(device)
    drift_history = []
    for r in range(1, rounds + 1):
        perm = torch.randperm(100)[:10].tolist()
        w_t = {k: v.clone() for k, v in m.state_dict().items()}
        updates = []
        for c in perm:
            cx, cy = c_dict[c]
            w_k = client_update(m, cx, cy, epochs=3, batch_size=20, lr=0.03, device=device)
            delta = torch.cat([(w_k[k] - w_t[k]).flatten() for k in w_t])
            updates.append(delta)
        sims = []
        for i in range(len(updates)):
            for j in range(i + 1, len(updates)):
                sim = F.cosine_similarity(updates[i].unsqueeze(0), updates[j].unsqueeze(0)).item()
                sims.append(sim)
        mean_sim = float(np.mean(sims))
        drift_history.append({'round': r, 'cosine_similarity': mean_sim})
        fedavg_round(m, c_dict, perm, epochs=3, batch_size=20, lr=0.03, device=device)
    return drift_history

iid_drift = measure_drift(client_data_100)
for d in iid_drift: d['partition'] = 'IID'
noniid_part = load_partition('data/pathological_k100.npz')
noniid_data = {c: (train_x[idx], train_y[idx]) for c, idx in noniid_part.items()}
noniid_drift = measure_drift(noniid_data)
for d in noniid_drift: d['partition'] = 'Pathological Non-IID'

pd.DataFrame(iid_drift + noniid_drift).to_csv('results/survey_client_drift.csv', index=False)
print('Drift measurement completed.')

Drift measurement completed.


Client updates under pathological non-IID data show substantially lower cosine similarity, demonstrating empirical client drift.